In [1]:
%load_ext jupyter_black

In [52]:
import spatialdata as sd
from pathlib import Path
import harpy as hp
import spatialdata_plot
import napari_spatialdata
from napari_spatialdata import Interactive
from tqdm import tqdm
import tempfile
from numpy.random import default_rng
import interscellar

RNG = default_rng(seed=0)

napari_spatialdata.constants.PROJECT_2_5D_SHAPES_TO_2D = False
napari_spatialdata.constants.PROJECT_3D_POINTS_TO_2D = False

In [53]:
out_path = Path.cwd() / "data"
sdata_path = out_path / "merfish_mouse_ileum.sdata.zarr"

sdata = sd.read_zarr(sdata_path)

bbox_coords = {
    "x": (2500, 2600),  # 100 pixels in x
    "y": (4500, 4600),  # 100 pixels in y
    "z": (0, 9),  # All z slices
}

print(f"\nQuerying bounding box: {bbox_coords}")
sdata_small = sd.bounding_box_query(
    sdata,
    axes=["x", "y", "z"],
    min_coordinate=[bbox_coords["x"][0], bbox_coords["y"][0], bbox_coords["z"][0]],
    max_coordinate=[bbox_coords["x"][1], bbox_coords["y"][1], bbox_coords["z"][1]],
    target_coordinate_system="global",
)

sdata = sdata_small

sdata.write(out_path / "merfish_mouse_ileum.sdata.zarr", overwrite=True)
sdata = sd.read_zarr(out_path / "merfish_mouse_ileum.sdata.zarr")


Querying bounding box: {'x': (2500, 2600), 'y': (4500, 4600), 'z': (0, 9)}


/Users/macbook/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12/functools.py:909: UserWarning: The object has `points` element. Depending on the number of points, querying MAY suffer from performance issues. Please consider filtering the object before calling this function by calling the `subset()` method of `SpatialData`.
  return dispatch(args[0].__class__)(*args, **kw)


ValueError: Cannot overwrite. The target path of the write operation is in use. Please save the data to a different location. 
Details: the target path contains one or more files that Dask use for backing elements in the SpatialData object.
Workaround: please see discussion here https://github.com/scverse/spatialdata/discussions/520 .

In [13]:
from interscellar.core.find_cell_neighbors_3d import build_cell_graph_database_3d

In [15]:
masks = sdata["dapi_labels"]["scale0"]["image"].data.compute()
pixels = sdata["stains"]["scale0"]["image"].data.compute()

In [16]:
instances = sd.get_element_instances(sdata["dapi_labels"]).to_numpy()
instances

array([1653, 4628], dtype=int32)

In [17]:
image = sdata["stains"]["scale0"]["image"].data

In [18]:
sd.get_element_instances(sdata["dapi_labels"])

Index([1653, 4628], dtype='int32')

In [41]:
df = sd.get_centroids(sdata["dapi_labels"]).compute()
df.reset_index(names="cell_id", inplace=True)
df["phenotype"] = RNG.choice(["a", "b", "c"], len(df))
print(df)
df.to_csv(out_path / "interscellar_temp.csv")

   cell_id            x            y         z phenotype
0     1653  2579.971545  4577.203252  6.884095         a
1     4628  2520.684932  4587.205479  6.884095         a


In [48]:
neighbor_table_df, adata = interscellar.find_cell_neighbors_3d(
    ome_zarr_path=str(sdata_path / 'labels' / 'dapi_labels'),
    metadata_csv_path=out_path / "interscellar_temp.csv",
    max_distance_um = 0.5,
    voxel_size_um = (0.56, 0.28, 0.28),
    centroid_prefilter_radius_um = 75.0,
    cell_id = 'cell_id',
    cell_type = 'phenotype',
    centroid_x = 'x',
    centroid_y = 'y',
    centroid_z = 'z',
    n_jobs = 1,
    return_connection = False,
)

InterSCellar: Surface-based Cell Neighbor Detection - 3D

1. Loading metadata from: /Users/macbook/embl/projects/basel/3d-spatial-workshop-2025/data/interscellar_temp.csv...
Loaded 2 cells
Step 1 completed in 0.00 seconds
db_path: /Users/macbook/embl/projects/basel/3d-spatial-workshop-2025/data/interscellar_temp_neighbor_graph.db
output_csv: /Users/macbook/embl/projects/basel/3d-spatial-workshop-2025/data/interscellar_temp_neighbors_3d.csv
output_anndata: /Users/macbook/embl/projects/basel/3d-spatial-workshop-2025/data/interscellar_temp_neighbors_3d.h5ad
Surfaces pickle path: /Users/macbook/embl/projects/basel/3d-spatial-workshop-2025/data/interscellar_temp_neighbor_graph_surfaces.pkl
Graph state pickle path: /Users/macbook/embl/projects/basel/3d-spatial-workshop-2025/data/interscellar_temp_neighbor_graph_graph_state.pkl

2. Building neighbor graph...
Parameters: max_distance=0.5μm, n_jobs=1
Mode: Touching cells + near-neighbors
Loading segmentation mask from: /Users/macbook/embl/proje

RuntimeError: Error in neighbor detection pipeline: Error loading ome-zarr file: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [ ]:
2